# 10.03 · Code-based checks & LLM-as-judge

**Chapter:** Evaluation · **Milestone:** M5

## Goals
- [ ] Write code-based assertions first (cheap, deterministic, no judge needed)
- [ ] Build an LLM judge: binary verdict + critique, specific rubric, structured output
- [ ] Detect judge biases (position bias in pairwise comparisons)
- [ ] **Calibrate** the judge against labelled examples before trusting it

## Resources
- [Eugene Yan – Evaluating the effectiveness of LLM-evaluators](https://eugeneyan.com/writing/llm-evaluators/) — free
- [Hamel Husain – Creating a LLM-as-a-Judge that drives business results](https://hamel.dev/blog/posts/llm-judge/) — free
- [Judging LLM-as-a-Judge (MT-Bench paper)](https://arxiv.org/abs/2306.05685) — free

In [ ]:
import json
import random
import re

import numpy as np
from pydantic import BaseModel, Field
from sklearn.metrics import cohen_kappa_score, confusion_matrix

from aieng import rag
from aieng.llm import client

c, MODEL = client()
EXTRA = {"reasoning_effort": "none"} if c.base_url.host == "localhost" else {}

## 1. Code-based checks: use them whenever you can
Format, length, required/forbidden content, valid JSON, citations that exist, SQL that runs. Deterministic, free, fast.

In [ ]:
def check_citations(answer, retrieved_ids):
    cited = {int(x) for x in re.findall(r"\[(\d+)\]", answer)}
    return {"has_citation": bool(cited), "citations_valid": cited <= set(retrieved_ids)}


def check_length(answer, max_words=120):
    return {"short_enough": len(answer.split()) <= max_words}


def check_no_leak(answer):
    return {"no_prompt_leak": "using only the sources" not in answer.lower()}


answer, hits = rag.answer("When was Abraham Lincoln born?", max_tokens=300)
print(answer)
print({**check_citations(answer, [p for p, _, _ in hits]), **check_length(answer), **check_no_leak(answer)})

## 2. An LLM judge for what code can't check
Good judge prompts: one criterion per judge, explicit definitions of pass/fail, a few labelled examples, **critique before verdict**, binary output.

In [ ]:
class Verdict(BaseModel):
    critique: str = Field(description="1-2 sentences explaining the decision, written BEFORE the verdict")
    correct: bool


JUDGE = """You are grading an answer to a question against a reference answer.
PASS (correct=true) if the answer states the same fact as the reference, even if phrased differently or with extra correct detail.
FAIL (correct=false) if it contradicts the reference, gives a different fact, or doesn't answer.

Question: {question}
Reference answer: {reference}
Answer to grade: {answer}"""


def judge(question, reference, answer, model=MODEL, cl=c):
    return cl.chat.completions.parse(model=model, temperature=0, max_tokens=300, response_format=Verdict, **EXTRA, messages=[{"role": "user",
        "content": JUDGE.format(question=question, reference=reference, answer=answer)}]).choices[0].message.parsed


print(judge("How long was Lincoln's formal education?", "18 months", "About a year and a half of schooling in total."))
print(judge("How long was Lincoln's formal education?", "18 months", "He studied for four years at a university."))

## 3. Calibrate the judge
A judge is a classifier (02.03) — measure it. We build a labelled set where we **know** the truth: for real questions, a correct answer
(the gold answer paraphrased in a sentence) and an incorrect one (another question's answer). Then compare judge verdicts with truth.

In [ ]:
rows = rag.eval_set()[:30]
rng = random.Random(0)
labelled = []
for i, r in enumerate(rows):
    labelled.append((r["question"], r["answer"], f"The answer is {r['answer']}.", True))
    wrong = rows[(i + 7) % len(rows)]["answer"]
    labelled.append((r["question"], r["answer"], f"I believe it's {wrong}.", False))
rng.shuffle(labelled)

y_true, y_pred, disagreements = [], [], []
for q, ref, ans, truth in labelled:
    v = judge(q, ref, ans)
    y_true.append(truth); y_pred.append(v.correct)
    if v.correct != truth:
        disagreements.append((q, ref, ans, truth, v.critique))

y_true, y_pred = np.array(y_true), np.array(y_pred)
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
print(f"agreement {np.mean(y_true == y_pred):.0%} | Cohen's kappa {cohen_kappa_score(y_true, y_pred):.2f}")
print(f"TPR (passes real passes) {tp / (tp + fn):.0%} | TNR (fails real fails) {tn / (tn + fp):.0%}")
for d in disagreements[:4]:
    print("  disagreement:", d)

This calibration set is *easy* (wrong answers are clearly wrong). Real calibration uses **your own human labels on real outputs** (10.01's CSV):
split into train (few-shot examples for the judge) / dev (iterate the judge prompt) / test (report TPR/TNR once). Report both rates — a judge that
passes everything has 50% accuracy here but a TNR of 0.

## 4. Pairwise comparison & position bias
"Which answer is better, A or B?" is often easier to judge than absolute quality — but judges favour a position (often the first).
Always evaluate both orders; count a win only if it's consistent.

In [ ]:
class Pairwise(BaseModel):
    reasoning: str
    winner: str = Field(description="'A' or 'B'")


def pairwise(question, a, b):
    return c.chat.completions.parse(model=MODEL, temperature=0, max_tokens=300, response_format=Pairwise, **EXTRA, messages=[{"role": "user", "content":
        f"Question: {question}\n\nAnswer A: {a}\n\nAnswer B: {b}\n\nWhich answer is more accurate and helpful? Reply with winner 'A' or 'B'."}]).choices[0].message.parsed.winner.strip().upper()[:1]


same_quality = [("What is the capital of Uruguay?", "Montevideo.", "The capital is Montevideo."),
                ("When was Lincoln born?", "1809.", "He was born in 1809."),
                ("What do kangaroos eat?", "Mostly grasses.", "They are mainly grazers that eat grass."),
                ("Who was Lincoln's wife?", "Mary Todd.", "Mary Todd Lincoln."),
                ("What language is spoken in Uruguay?", "Spanish.", "Spanish is the official language.")]
first_wins = 0
for q, a, b in same_quality:
    w1, w2 = pairwise(q, a, b), pairwise(q, b, a)
    first_wins += (w1 == "A") + (w2 == "A")
    print(f"{q:40} order1 -> {w1} | swapped -> {w2} | {'consistent' if w1 != w2 else 'POSITION-DEPENDENT'}")
print(f"first position chosen {first_wins}/{2 * len(same_quality)} times (unbiased ≈ 50% on equal-quality pairs)")

Other known judge biases: **verbosity** (longer looks better), **self-preference** (a model prefers its own outputs), and sensitivity to
formatting. Mitigations: swap positions, use a different model family as judge, length-controlled comparisons, and calibration.

## 5. Practical rules
1. Code checks first; LLM judges only for genuinely fuzzy criteria.
2. One judge per failure mode from your taxonomy (10.01), each binary with a critique.
3. Use the **strongest affordable model** as judge, and a different one than the system under test.
4. Calibrate on human labels; re-check when you change the judge prompt or model.

## Exercises
1. Calibrate the judge with a hosted model (`client("gemini")` or `client("groq")`). How much better are TPR/TNR than the 1.7B judge?
2. Make the calibration set harder: near-miss wrong answers (off-by-one years, the right person's wrong attribute). Re-measure.
3. Use your 10.01 annotations: write a judge for your biggest failure category and measure its agreement with your labels.

In [ ]:
# your code here